# Inspect Consolidated Recharge Datasets

Load and visualize the three recharge source datasets at the annual time-step for year 2000.

Sources (see `catalog/variables.yml` → `recharge`):
- Reitz 2017 (`total_recharge`, m/year) — already annual
- WaterGAP 2.2d (`qrdif`, kg m-2 s-1) — monthly, summed to annual on the fly
- ERA5-Land (`ssro`, m/month) — monthly, summed to annual on the fly

Note: monthly sources are summed to annual here so issues in any one month surface in the annual cell. The actual recharge target builder does the same aggregation downstream.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")

TARGET_YEAR = 2000


## Load all three datasets

In [ ]:
datasets = {
    "Reitz 2017 (total_recharge)": {
        "path": DATASTORE / "reitz2017" / "reitz2017_consolidated.nc",
        "var": "total_recharge",
        "time_step": "annual",
        "units": "m/year",
    },
    "WaterGAP 2.2d (qrdif)": {
        "path": DATASTORE / "watergap22d" / "watergap22d_qrdif_cf.nc",
        "var": "qrdif",
        "time_step": "monthly",
        "units": "kg m-2 s-1 (summed to annual)",
    },
    "ERA5-Land (ssro)": {
        "path": DATASTORE / "era5_land" / "monthly" / f"era5_land_monthly_{TARGET_YEAR}.nc",
        "var": "ssro",
        "time_step": "monthly",
        "units": "m (summed to annual)",
    },
}

opened = {}
for label, info in datasets.items():
    nc_path = info["path"]
    if not nc_path.exists():
        print(f"SKIP {label}: {nc_path} not found (run fetch first)")
        continue
    ds = xr.open_dataset(nc_path)
    opened[label] = (ds, info)
    print(f"{label}: {list(ds.data_vars)} | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}")


## Dataset representations

In [ ]:
for label, (ds, _) in opened.items():
    print(f"{'=' * 60}\n{label}\n{'=' * 60}")
    display(ds)


## Plot annual values for target year

In [ ]:
n = len(opened)
if n == 0:
    print("No datasets available yet. Run the fetch commands first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    axes = axes.flatten() if n > 1 else [axes]

    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = axes[idx]
        var = info["var"]
        if info["time_step"] == "monthly":
            da = ds[var].sel(
                time=slice(f"{TARGET_YEAR}-01", f"{TARGET_YEAR}-12")
            ).sum("time")
            label_time = f"{TARGET_YEAR} (sum of 12 months)"
        else:
            da = ds[var].sel(time=str(TARGET_YEAR), method="nearest")
            label_time = str(da.time.values)[:10] if da.time.size else str(TARGET_YEAR)

        da.plot(ax=ax, cmap="BuGn", robust=True)
        ax.set_title(f"{label}\n{label_time} | {info['units']}", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    for idx in range(len(opened), len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(f"Recharge — annual values for {TARGET_YEAR}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


## Normalized comparison (mm/year, ERA5-Land footprint)

Convert all three sources to **mm/year** and clip to the **ERA5-Land bounding box** for a like-for-like comparison.

The native units differ:

- Reitz 2017 `total_recharge` is already annual in inches/year. Multiply by 25.4 to get mm/year.
- WaterGAP 2.2d `qrdif` is a monthly mean rate in kg m⁻² s⁻¹ (= mm/s for water). Multiply each month by its number of seconds to get mm/month, then sum 12 months.
- ERA5-Land `ssro` is a monthly accumulated depth in m (`time: sum`). Sum 12 months and multiply by 1000 to get mm/year.

ERA5-Land is already subset to CONUS plus contributing watersheds, so using its lat/lon range as the clip extent gives the same geographic footprint for all three panels and keeps the colour scale dominated by CONUS values rather than tropical maxima from WaterGAP's global grid.


In [ ]:
import calendar

import numpy as np


def _to_mm_per_year(ds, info, year):
    """Convert a source-specific recharge variable to a 2D field in mm/year."""
    var = info["var"]
    units = info["units"]

    if units == "m/year":
        # Reitz 2017: annual m/year  ->  mm/year
        da = ds[var].sel(time=str(year), method="nearest")
        return da * 1000.0

    if "kg m-2 s-1" in units:
        # WaterGAP qrdif: monthly mean rate (mm/s).  Weight each month by its
        # length in seconds, then sum to a yearly total.
        monthly = ds[var].sel(time=slice(f"{year}-01", f"{year}-12"))
        secs_per_month = np.array(
            [calendar.monthrange(year, m)[1] * 86400 for m in range(1, 13)],
            dtype="float64",
        )
        weights = xr.DataArray(secs_per_month, coords=[monthly.time], dims=["time"])
        return (monthly * weights).sum("time")

    if units.startswith("m "):
        # ERA5-Land ssro: monthly accumulated depth in m.  Sum 12 months -> m/yr,
        # then * 1000 -> mm/yr.
        monthly = ds[var].sel(time=slice(f"{year}-01", f"{year}-12"))
        return monthly.sum("time") * 1000.0

    raise ValueError(f"Don't know how to normalize units {units!r}")


def _clip_to_bbox(da, lon_range, lat_range):
    """Clip a 2D DataArray to a lon/lat bounding box, handling descending lats."""
    lon_dim = "lon" if "lon" in da.dims else "x"
    lat_dim = "lat" if "lat" in da.dims else "y"
    lat_vals = da[lat_dim].values
    if lat_vals[0] > lat_vals[-1]:  # descending
        lat_slice = slice(lat_range[1], lat_range[0])
    else:
        lat_slice = slice(lat_range[0], lat_range[1])
    return da.sel({lon_dim: slice(*lon_range), lat_dim: lat_slice})


# Reference dataset providing the clip footprint and colour scale
_REF_LABEL = "ERA5-Land (ssro)"

if opened:
    # 1) Normalize all sources to mm/year
    norm = {}
    for label, (ds, info) in opened.items():
        norm[label] = _to_mm_per_year(ds, info, TARGET_YEAR)

    # 2) Use ERA5-Land's lat/lon range as the clip footprint
    if _REF_LABEL in norm:
        ref = norm[_REF_LABEL]
        lon_min, lon_max = float(ref.lon.min()), float(ref.lon.max())
        lat_min, lat_max = float(ref.lat.min()), float(ref.lat.max())
        print(f"ERA5-Land footprint: lon [{lon_min:.2f}, {lon_max:.2f}]  lat [{lat_min:.2f}, {lat_max:.2f}]")
        clipped = {label: _clip_to_bbox(da, (lon_min, lon_max), (lat_min, lat_max)) for label, da in norm.items()}
    else:
        print(f"WARN: {_REF_LABEL} not loaded; skipping ERA5-footprint clip")
        clipped = norm

    for label, da in clipped.items():
        print(f"{label}: mean = {float(da.mean(skipna=True)):.2f} mm/year (ERA5 footprint)")

    # 3) Shared colour scale derived from ERA5 (land-masked CONUS)
    if _REF_LABEL in clipped:
        ref_vals = clipped[_REF_LABEL].values.ravel()
        ref_vals = ref_vals[~np.isnan(ref_vals)]
        vmin = float(np.percentile(ref_vals, 2))
        vmax = float(np.percentile(ref_vals, 98))
        print(f"Colour scale from {_REF_LABEL}: {vmin:.2f} - {vmax:.2f} mm/year")
    else:
        all_vals = np.concatenate([c.values.ravel() for c in clipped.values()])
        all_vals = all_vals[~np.isnan(all_vals)]
        vmin = float(np.percentile(all_vals, 2))
        vmax = float(np.percentile(all_vals, 98))

    fig, axes = plt.subplots(1, len(clipped), figsize=(20, 6), squeeze=False)
    for idx, (label, da) in enumerate(clipped.items()):
        ax = axes[0, idx]
        da.plot(ax=ax, cmap="BuGn", vmin=vmin, vmax=vmax)
        ax.set_title(f"{label}\nmm/year - {TARGET_YEAR}", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    fig.suptitle(
        f"Recharge - mm/year, ERA5-Land footprint - {TARGET_YEAR}\n"
        f"colour scale from {_REF_LABEL}",
        fontsize=13, y=1.03,
    )
    plt.tight_layout()
    plt.show()
else:
    print("No datasets available yet.")


### Why do the three recharge sources disagree?

Observed CONUS-wide annual means (ERA5-Land footprint, year 2000), with corrected Reitz units:

- Reitz 2017 `total_recharge`: ~122 mm/year (raw m/year × 1000)
- WaterGAP 2.2d `qrdif`: 54 mm/year
- ERA5-Land `ssro`: 120 mm/year

**These products do not measure the same thing.** Each estimates a different hydrologic flux that we treat as a recharge proxy:

- **Reitz 2017 `total_recharge`** is an *empirical* regression estimate of total groundwater recharge over CONUS, calibrated against base-flow and chloride-mass-balance observations. It is intended to capture long-term mean recharge, not year-to-year variability. The Reitz target in TM 6-B10 is used for *relative* year-to-year change rather than absolute magnitude. Reitz 2017 reports a CONUS mean of approximately 162 mm/year.
- **WaterGAP 2.2d `qrdif`** is *diffuse groundwater recharge* simulated by the WaterGAP global hydrology model: water that bypasses surface runoff and percolates to a groundwater store. WaterGAP partitions soil-water output into surface runoff, diffuse recharge (`qrdif`), and focused recharge; only `qrdif` is used here.
- **ERA5-Land `ssro`** is *sub-surface runoff* from the ECMWF land-surface scheme: drainage out the bottom of the soil column, which feeds groundwater and slow-response stream flow. It is a *proxy* for recharge - not formally equivalent.

**Why the numbers diverge**

1. **Different definitions.** ssro is total sub-surface drainage (which over time fills aquifers and emerges as base flow), whereas qrdif is specifically the fraction routed to a groundwater store. ssro therefore tends to be larger than qrdif under the same climate forcing.

2. **Different forcing and land-surface physics.** WaterGAP is forced by WFDEI-GPCC (a bias-corrected reanalysis-based product); ERA5-Land is the ECMWF reanalysis itself. The two land-surface models have different soil hydrology, snowmelt, and PET schemes, which produces different partitioning between runoff and drainage.

3. **Different conceptualization of recharge in arid regions.** Empirical estimates and process models often disagree in arid or seasonally dry regions because actual recharge is rare, episodic, and small relative to gross fluxes. Process models tend to overestimate recharge there if subsurface storage is poorly parameterised.

**Calibration target implication**

For NHM, the recharge target uses Reitz, WaterGAP, and ERA5-Land as a *normalized* range (min-max) rather than absolute values - the calibration is sensitive to relative spatial pattern, not absolute magnitude.

**Reitz units correction**

An earlier version of `catalog/sources.yml` declared Reitz units as `inches/year`, which produced values ~8x too low when converted with × 25.4. The ScienceBase landing page documents the rasters as **m/year**, and the source GeoTIFFs themselves carry no embedded unit metadata (`Band.GetUnitType()`, file/band `Metadata`, and PAM `.aux.xml` sidecars are all empty). The catalog has been corrected to `m/year`; this notebook now applies × 1000 to convert raw m/year to mm/year. Cross-checks confirm the m/year interpretation: CONUS mean of 122 mm/year is in the ballpark of the published 162 mm/year, and spot magnitudes at the Olympic Peninsula (~1160 mm/year), western Cascades (~530 mm/year), Iowa cropland (~92 mm/year) and the desert SW (single-digit mm/year) all line up with expected geographic patterns.


## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()
